In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT / "src"))

CONFIG_PATH = "experiments/qwen3vl/qwen3vl_2b_zero_shot_baseline/config.toml"
LIMIT = None  

from shield.config import load_and_validate, method

config = load_and_validate(PROJECT_ROOT / CONFIG_PATH)
print("Esperimento:", config["experiment"]["name"], "| method:", method(config),
      "| dataset:", config["dataset"]["version"])

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Nessuna GPU disponibile. Verifica CUDA sul server.")
print(torch.cuda.get_device_name(0))

adapter_dir = None
if method(config) != "none":
    adapter_dir = PROJECT_ROOT / "outputs" / config["experiment"]["family"] / config["experiment"]["name"] / "final"
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Adapter non trovato in {adapter_dir}.")
print("adapter:", adapter_dir)

In [ ]:
from shield.evaluation.pipeline import run_evaluation

results = run_evaluation(config, PROJECT_ROOT, adapter_dir=adapter_dir, limit=LIMIT)

print("\n=== Metriche aggregate ===")
for key, value in results["aggregate"].items():
    if isinstance(value, (int, float)):
        print(f"  {key:24s} {value:.4f}")
if results.get("n_skipped"):
    print(f"\n⚠️ {results['n_skipped']} esempi saltati (vedi 'skipped' in metrics.json): test set effettivo ridotto.")

comparison = results.get("comparison_vs_baseline")
if isinstance(comparison, dict) and "status" not in comparison:
    print("\n=== Confronto vs baseline (delta) ===")
    for key, entry in comparison.items():
        print(f"  {key:24s} {entry['baseline']:.4f} -> {entry['current']:.4f}  (Δ {entry['delta']:+.4f})")
elif isinstance(comparison, dict):
    print("\n[eval]", comparison["status"])

significance = results.get("significance")
if isinstance(significance, dict) and significance.get("metrics"):
    print(f"\n=== Significatività (paired bootstrap, n={significance.get('n_common')}, resamples={significance.get('n_resamples')}) ===")
    for key, entry in significance["metrics"].items():
        if "significant" in entry:
            verdict = "SIGNIFICATIVO" if entry["significant"] else "non significativo"
            print(f"  {key:24s} Δ {entry['delta']:+.4f}  IC95 [{entry['ci95_low']:+.4f}, {entry['ci95_high']:+.4f}]  p={entry['p_value']:.4f} -> {verdict}")
elif isinstance(significance, dict) and "status" in significance:
    print("\n[eval] significatività:", significance["status"])

print("\nRisultati salvati in:", results["output_dir"])

In [ ]:
from shield.tracking import log_artifact_if_exists, log_numeric_metrics, mlflow_run

with mlflow_run(config, root=PROJECT_ROOT):
    log_numeric_metrics(results["aggregate"], prefix="eval")
    if results.get("disaggregated"):
        log_numeric_metrics(results["disaggregated"], prefix="eval.by")
    if results.get("operational"):
        log_numeric_metrics(results["operational"], prefix="eval.op")
    comparison = results.get("comparison_vs_baseline")
    if isinstance(comparison, dict) and "status" not in comparison:
        log_numeric_metrics(comparison, prefix="eval.cmp")
    if isinstance(results.get("significance"), dict) and results["significance"].get("metrics"):
        log_numeric_metrics(results["significance"]["metrics"], prefix="eval.sig")
    log_artifact_if_exists(str(results["output_dir"]), artifact_path="evaluation", allow_dir=True)
    log_artifact_if_exists(str(PROJECT_ROOT / CONFIG_PATH), artifact_path="config")
print("Loggato su MLflow.")